In [1]:
import pandas as pd

In [3]:
# print current pwd
import os
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\AN\Collection\Learn\HUST\Intro to DS\Project\DS-project-20251\src\preprocessing


In [ ]:
df = pd.read_json("../../data/merged/merged.jsonl",lines=True)

print(df.head())
print(len(df))
# check for duplicate hotel_id and print total number of unique hotel id
duplicate_hotel_ids = df[df.duplicated(subset=['hotel_id'], keep=False)]['hotel_id'].unique()
print(f"Duplicate hotel IDs: {duplicate_hotel_ids}")

# assign all hotel with duplicate hotel_id to a new dataframe
duplicate_hotels_df = df[df['hotel_id'].isin(duplicate_hotel_ids)].copy()
print(f"Total duplicate hotels: {len(duplicate_hotels_df)}")

   hotel_id                                         hotel_name  \
0  71897952                       Nha Nghi Nhung - Nhung Motel   
1  49685029           Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa)   
2  65481766    Baly Hotel Bà Rịa City (Baly Hotel Ba Ria City)   
3   5808626  Citadines Central Bình Dương (Citadines Centra...   
4  10613675                  KHÁCH SẠN DIAMOND (DIAMOND HOTEL)   

                                       hotel_address  \
0         73 Đoàn Thị Điểm, Bà Rịa, Bà Rịa, Việt Nam   
1  KDC Baria City Gate, Long Huong Ward, Ba Ria C...   
2                     QL51, Bà Rịa, Bà Rịa, Việt Nam   
3  Số 328C, Đại lộ B nh Dương, Khu phố Hưng Lộc, ...   
4  153 Hoang Van Thu Street, Thủ Dầu Một, Bình Dư...   

                                          room_types      region  
0  [{'room_type_name': 'Phòng Deluxe Có Giường Cỡ...      Bà Rịa  
1  [{'room_type_name': 'Phòng Tiêu Chuẩn (Standar...      Bà Rịa  
2  [{'room_type_name': 'Phòng Có Giường Cỡ King V...      Bà Rịa 

In [8]:
print(duplicate_hotels_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 162 entries, 3690 to 3724
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   hotel_id       162 non-null    int64 
 1   hotel_name     160 non-null    object
 2   hotel_address  160 non-null    object
 3   room_types     162 non-null    object
 4   region         162 non-null    object
dtypes: int64(1), object(4)
memory usage: 7.6+ KB
None


In [12]:
duplicate_hotels_df["room_count"] = duplicate_hotels_df["room_types"].apply(lambda x: len(x) if isinstance(x, list) else 0)

In [15]:
# aggregate by hotel_id, for each hotel_id, keep the row with the maximum room_count, if tie, keep the first one
aggregated_df = duplicate_hotels_df.loc[duplicate_hotels_df.groupby('hotel_id')['room_count'].idxmax()]
print(f"Total aggregated hotels: {len(aggregated_df)}")
print(aggregated_df.info())

Total aggregated hotels: 81
<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 3690 to 3846
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   hotel_id       81 non-null     int64 
 1   hotel_name     80 non-null     object
 2   hotel_address  80 non-null     object
 3   room_types     81 non-null     object
 4   region         81 non-null     object
 5   room_count     81 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 4.4+ KB
None


In [18]:
no_duplicate_df = df[~df['hotel_id'].isin(duplicate_hotel_ids)].copy()

In [19]:
deduplicate_df = pd.concat([no_duplicate_df, aggregated_df], ignore_index=True)

In [20]:
print(deduplicate_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4446 entries, 0 to 4445
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hotel_id       4446 non-null   int64  
 1   hotel_name     4271 non-null   object 
 2   hotel_address  4293 non-null   object 
 3   room_types     4446 non-null   object 
 4   region         4446 non-null   object 
 5   room_count     81 non-null     float64
dtypes: float64(1), int64(1), object(4)
memory usage: 208.5+ KB
None


In [21]:
null_df = deduplicate_df[(deduplicate_df['hotel_name'].isnull()) | (deduplicate_df['hotel_address'].isnull())]
print(len(null_df))

175


In [23]:
null_df.to_csv("../../data/processed/hotels_with_null_name_or_address.csv", index=False)

In [26]:
deduplicate_df.drop(columns=['room_count'], inplace=True)
# drop null in deduplicate_df
deduplicate_df = deduplicate_df.dropna(subset=['hotel_name', 'hotel_address'])

In [27]:
print(deduplicate_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 4271 entries, 0 to 4445
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   hotel_id       4271 non-null   int64 
 1   hotel_name     4271 non-null   object
 2   hotel_address  4271 non-null   object
 3   room_types     4271 non-null   object
 4   region         4271 non-null   object
dtypes: int64(1), object(4)
memory usage: 200.2+ KB
None


In [28]:
hotel_general_info_df = deduplicate_df[['hotel_id', 'hotel_name', 'hotel_address', 'region']]

In [30]:
print(hotel_general_info_df.head())

   hotel_id                                         hotel_name  \
0  71897952                       Nha Nghi Nhung - Nhung Motel   
1  49685029           Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa)   
2  65481766    Baly Hotel Bà Rịa City (Baly Hotel Ba Ria City)   
3   5808626  Citadines Central Bình Dương (Citadines Centra...   
4  10613675                  KHÁCH SẠN DIAMOND (DIAMOND HOTEL)   

                                       hotel_address      region  
0         73 Đoàn Thị Điểm, Bà Rịa, Bà Rịa, Việt Nam      Bà Rịa  
1  KDC Baria City Gate, Long Huong Ward, Ba Ria C...      Bà Rịa  
2                     QL51, Bà Rịa, Bà Rịa, Việt Nam      Bà Rịa  
3  Số 328C, Đại lộ B nh Dương, Khu phố Hưng Lộc, ...  Bình Dương  
4  153 Hoang Van Thu Street, Thủ Dầu Một, Bình Dư...  Bình Dương  


In [31]:
hotel_general_info_df.to_csv("../../data/processed/hotel_general_info.csv", index=False)

In [36]:
room_types_df = deduplicate_df[['hotel_id', 'room_types']].copy()
room_types_df = room_types_df.explode('room_types').reset_index(drop=True)
room_types_expanded = pd.json_normalize(room_types_df['room_types'])
room_types_expanded.columns = [f"room_{col}" for col in room_types_expanded.columns]
room_types_final_df = pd.concat([room_types_df[['hotel_id']], room_types_expanded], axis=1)
print(room_types_final_df.head())

   hotel_id                                room_room_type_name  \
0  71897952  Phòng Deluxe Có Giường Cỡ King (Deluxe King Room)   
1  49685029                   Phòng Tiêu Chuẩn (Standard Room)   
2  49685029  Phòng gia đình có ban công (Family Room with B...   
3  65481766  Phòng Có Giường Cỡ King Với Ban Công (King Roo...   
4   5808626          Phòng Studio Executive (Studio Executive)   

                                      room_amenities  \
0  [{'amenity': 'Diện tích phòng: 18 m²', 'type':...   
1  [{'amenity': 'Diện tích phòng: 30 m²', 'type':...   
2  [{'amenity': 'Diện tích phòng: 45 m²', 'type':...   
3  [{'amenity': 'Diện tích phòng: 20 m²', 'type':...   
4  [{'amenity': 'Diện tích phòng: 35 m²', 'type':...   

                                  room_price_options  
0  [{'option_name': 'Thông tin sức chứa phòng: 2 ...  
1  [{'option_name': 'Thông tin sức chứa phòng: 2 ...  
2  [{'option_name': 'Thông tin sức chứa phòng: 4 ...  
3  [{'option_name': 'Thông tin sức chứa phòng:

In [37]:
room_types_final_df["room_type_id"] = room_types_final_df.index + 1  # start from 1

In [38]:
room_types_final_df[["hotel_id", "room_room_type_name", "room_type_id"]].to_csv("../../data/processed/hotel_room_types.csv", index=False)

In [40]:
room_amenities_df = room_types_final_df[['room_type_id','room_amenities']].to_csv("../../data/processed/room_amenities.csv", index=False)

In [48]:
price_df = room_types_final_df[['room_type_id', 'room_price_options']].copy()
price_df = price_df.explode('room_price_options').reset_index(drop=True)

price_option_expanded = pd.json_normalize(price_df["room_price_options"])
price_option_expanded.columns = [f"price_{col}" for col in price_option_expanded.columns]
price_final_df = pd.concat([price_df[['room_type_id']], price_option_expanded], axis=1)

price_final_df.to_csv("../../data/processed/room_price_options.csv", index=False)